# CrocoDashChecks

Checks that CrocoDash works end to end on NCAR's systems. Every week it builds real CESM
cases with CrocoDash and runs them on Derecho; every day it checks that every data source
CrocoDash reads still works. Both test the latest CrocoDash `main`. This page shows where
each check stands.

- [How it works](how_it_works.md): what each check does, when it runs, and how to run it
  yourself.
- [Performance](performance.ipynb): how long CrocoDash's expensive steps take, with fits for
  sizing jobs.

```{toctree}
:hidden:
performance
how_it_works
cesm_runs
```

In [ ]:
import checks
from checks import FAIL, NOT_RUN, OK, WARN

runs = checks.load(checks.CESM_RUNS)  # the weekly CESM runs
health = checks.load(checks.HEALTH)  # the daily data-access check

In [ ]:
def summary_row(name, results, counts, word, kind):
    """The check, its result, when it last ran, and the CrocoDash it tested."""
    if results is None:
        return [name, f"{WARN} No results yet", "", ""]
    ok, total = counts(results)
    if ok == total:
        result = f"{OK} {ok}/{total} {word}"
    else:
        result = f"{FAIL} {ok}/{total} {word}: {total - ok} failing"
    if checks.is_stale(results["date"], kind):
        days = checks.age_days(results["date"])
        result += f" ({WARN} stale: {days} days old, so its schedule may have stopped)"
    crocodash = checks.commit(results.get("crocodash"), "CrocoDash")
    return [name, result, checks.when(results["date"]), crocodash]


checks.show(
    checks.table(
        ["Check", "Result", "Last run", "CrocoDash tested"],
        [
            summary_row(
                "[Weekly CESM runs](#cesm)",
                runs,
                checks.cesm_counts,
                "tests passing",
                "cesm",
            ),
            summary_row(
                "[Data access](#data-access)",
                health,
                checks.health_counts,
                "access methods OK",
                "health",
            ),
        ],
    )
)

(cesm)=
## Weekly CESM runs on Derecho

Every Monday, each regional SMS test in MOM_interface's `testlist_mom.xml` is built with
CrocoDash on a small test domain and run in CESM on one Derecho node. Each test passes through
CIME's phases in order, from creating the case to running the model. More:
[Weekly CESM runs](cesm_runs.md); the code is in
[`cesm_runs/`](https://github.com/CROCODILE-CESM/CrocoDashChecks/tree/main/cesm_runs).

In [ ]:
# CESM's stub components, left out of the components column.
STUB_COMPONENTS = {
    "SATM",
    "SLND",
    "SICE",
    "SOCN",
    "SROF",
    "SGLC",
    "SWAV",
    "SESP",
    "SIAC",
}


def components(longname):
    """A compset's active components, e.g. 'DATM (JRA) · CICE (PRES) · MOM6'."""
    active = []
    for token in longname.split("_")[1:]:
        name, *mods = token.split("%")
        mods = [m for m in mods if m != "REGIONAL"]
        if name not in STUB_COMPONENTS:
            active.append(f"{name} ({', '.join(mods)})" if mods else name)
    return " · ".join(active)


def phase_marks(test):
    """One mark per CIME phase, in order: passed, failed or not reached."""
    status = dict(test["phases"])
    marks = {"PASS": OK, "FAIL": FAIL}
    return "".join(marks.get(status.get(phase), NOT_RUN) for phase, _ in checks.PHASES)


def result(test):
    """'Pass', or the first phase that failed."""
    if test["passed"]:
        return f"{OK} Pass"
    status = dict(test["phases"])
    failed = [label for phase, label in checks.PHASES if status.get(phase) == "FAIL"]
    return f"{FAIL} Failed: {failed[0]}" if failed else f"{WARN} Incomplete"


if runs is None:
    checks.show("No results yet.")
else:
    tests = sorted(
        runs["tests"], key=lambda t: (t["passed"], t["compset"])
    )  # failures first
    rows = [
        [
            f"`{t['compset']}`",
            f"`{t['test']}`",
            components(t["longname"]),
            phase_marks(t),
            result(t),
        ]
        for t in tests
    ]
    order = " → ".join(label for _, label in checks.PHASES)
    checks.show(
        checks.table(["Compset", "Test", "Components", "Phases", "Result"], rows)
        + f"\n\nPhases, left to right: {order}. {OK} passed, {FAIL} failed, {NOT_RUN} not reached."
        + f"\n\nRun {checks.when(runs['date'])} with CESM {checks.commit(runs.get('cesm'), 'CESM')}"
        + f" and CrocoDash {checks.commit(runs.get('crocodash'), 'CrocoDash')}."
    )

(data-access)=
## Data access, daily on GLADE

Every day, each way CrocoDash can fetch data (every access method of every product in its
`ProductRegistry`) is called for real on Casper, with GLADE mounted and our own credentials. A
method that fails says why; one still running after 15 minutes is stopped and counted as
failing. The code is in
[`data_access/`](https://github.com/CROCODILE-CESM/CrocoDashChecks/tree/main/data_access).

In [ ]:
if health is None:
    checks.show("No results yet.")
else:
    ok, total = checks.health_counts(health)
    by_name = sorted(
        health["validate_checks"], key=lambda c: (c["product"], c["method"])
    )
    failing = [c for c in by_name if not c["ok"]]
    working = [c for c in by_name if c["ok"]]
    text = (
        f"{ok} of {total} access methods working. Checked {checks.when(health['date'])}"
        f" with CrocoDash {checks.commit(health.get('crocodash'), 'CrocoDash')}."
    )
    if failing:
        text += f"\n\n**{FAIL} Failing ({len(failing)})**\n\n" + checks.table(
            ["Product", "Access method", "Why it failed", "Time"],
            [
                [
                    c["product"],
                    f"`{c['method']}`",
                    checks.plain(c.get("detail", "")),
                    checks.fmt_s(c["elapsed_s"]),
                ]
                for c in failing
            ],
        )
    text += f"\n\n**{OK} Working ({len(working)})**\n\n" + checks.table(
        ["Product", "Access method", "Time"],
        [
            [c["product"], f"`{c['method']}`", checks.fmt_s(c["elapsed_s"])]
            for c in working
        ],
    )
    text += f"\n\nRaw results: [health.json]({checks.SITE_URL}/health.json)"
    checks.show(text)

(performance-summary)=
(mom6-scaling)=
## Performance

How long CrocoDash's expensive steps take and how much memory they need, with fits for sizing
jobs: open boundaries, GEBCO bathymetry, GLOFAS runoff mapping, and MOM6's speed against its
task count. See the [Performance page](performance.ipynb).